# 🧠 02 — Extração de Features DINOv2 e Treinamento do Classificador

Neste notebook vamos:
1. Carregar o backbone DINOv2 ViT-B/14 com aceleração CUDA.
2. Extrair embeddings de 768 dimensões para todas as imagens.
3. Treinar classificadores (Logistic Regression, SVM, Random Forest).
4. Avaliar com validação cruzada, matriz de confusão e relatório de métricas.
5. Salvar o melhor modelo para uso posterior.

---
## 1. Dependências

In [ ]:
from __future__ import annotations

import sys
import logging
import time
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sys.path.insert(0, str(Path.cwd().parent))

from src.config import DINOv2Config
from src.transforms import build_transforms
from src.dataset import ThermalImageDataset
from src.feature_extractor import DINOv2FeatureExtractor

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Configuração e Carregamento do Dataset

In [ ]:
config = DINOv2Config(
    data_dir=Path("../data/raw"),
    image_size=224,
    batch_size=32,
    num_workers=0,  # 0 para notebooks
    device="cuda",  # mude para 'cpu' se não tiver GPU
)

transform = build_transforms(config)
dataset = ThermalImageDataset(config, transform=transform)

print(f"Dataset: {len(dataset)} imagens, {len(dataset.classes)} classes")
print(f"Classes: {dataset.classes}")

---
## 3. Extração de Features DINOv2

O DINOv2 ViT-B/14 gera um embedding de **768 dimensões** para cada imagem,
capturando representações visuais de alto nível sem necessidade de fine-tuning.

In [ ]:
extractor = DINOv2FeatureExtractor(config)

start = time.time()
features, labels, paths = extractor.extract_from_dataset(dataset)
elapsed = time.time() - start

print(f"\nExtração concluída em {elapsed:.1f}s")
print(f"Features shape: {features.shape}")
print(f"Labels shape: {labels.shape}")

In [ ]:
# Salvar features extraídas para reutilização futura
output_dir = Path("../outputs/features")
output_dir.mkdir(parents=True, exist_ok=True)

np.save(output_dir / "features_dinov2.npy", features)
np.save(output_dir / "labels.npy", labels)

print(f"✅ Features salvas em {output_dir}/")

---
## 4. Visualização das Features (t-SNE)

Projeção 2D das features DINOv2 para verificar separabilidade entre classes.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(features)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    features_2d[:, 0], features_2d[:, 1],
    c=labels, cmap="viridis", alpha=0.7, s=30
)
ax.set_title("t-SNE das Features DINOv2 (768-dim → 2D)", fontsize=14)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")

handles = [
    plt.scatter([], [], c=plt.cm.viridis(i / (len(dataset.classes) - 1)), s=50, label=name)
    for i, name in enumerate(dataset.classes)
]
ax.legend(handles=handles, title="Classes", loc="best")
plt.tight_layout()
plt.show()

---
## 5. Treinamento e Avaliação dos Classificadores

Comparamos três algoritmos clássicos usando **Validação Cruzada Estratificada (5-fold)**.

In [ ]:
classifiers = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42)),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, pipeline in classifiers.items():
    scores = cross_val_score(pipeline, features, labels, cv=cv, scoring="accuracy")
    results[name] = scores
    print(f"{name:25s} → Acurácia: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Gráfico comparativo dos classificadores
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(
    [results[name] for name in results],
    labels=list(results.keys()),
    patch_artist=True,
    boxprops=dict(facecolor="lightblue"),
)
ax.set_title("Comparativo de Classificadores (5-Fold CV)", fontsize=14)
ax.set_ylabel("Acurácia")
ax.set_ylim(0.8, 1.05)
plt.tight_layout()
plt.show()

---
## 6. Treinamento Final e Matriz de Confusão

Treinamos o melhor classificador no dataset completo e avaliamos com a matriz de confusão.

In [ ]:
# Selecionar o melhor classificador automaticamente
best_name = max(results, key=lambda k: results[k].mean())
best_pipeline = classifiers[best_name]
print(f"Melhor classificador: {best_name} ({results[best_name].mean():.4f})")

# Treinar no dataset completo
best_pipeline.fit(features, labels)
predictions = best_pipeline.predict(features)

# Relatório de classificação
print("\n" + classification_report(
    labels, predictions,
    target_names=dataset.classes
))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(labels, predictions)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=dataset.classes,
    yticklabels=dataset.classes, ax=ax
)
ax.set_title(f"Matriz de Confusão — {best_name}", fontsize=14)
ax.set_xlabel("Predição")
ax.set_ylabel("Real")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

---
## 7. Salvar o Modelo Treinado

In [ ]:
model_dir = Path("../outputs/models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "classifier_dinov2.pkl"

model_artifact = {
    "pipeline": best_pipeline,
    "classifier_name": best_name,
    "classes": dataset.classes,
    "feature_dim": features.shape[1],
    "train_accuracy": accuracy_score(labels, predictions),
    "cv_accuracy_mean": results[best_name].mean(),
    "cv_accuracy_std": results[best_name].std(),
}

with open(model_path, "wb") as f:
    pickle.dump(model_artifact, f)

print(f"✅ Modelo salvo em {model_path}")
print(f"   Classificador: {best_name}")
print(f"   Acurácia CV: {results[best_name].mean():.4f} ± {results[best_name].std():.4f}")